# 07 — UMAP (direction + hyp + kappa)

Requires `pip install umap-learn`. One UMAP fit on **all** rows in `cache/merged.parquet`, then **four** views (hyperbolic radius, hexbin density, κ, graph degree). Full *N* can be slow and memory-heavy.


### UMAP on direction + radial scalars

**Method:** One UMAP fit (`random_state=42`, `n_neighbors=30`, `min_dist=0.1`, **PCA init**, `n_jobs=1`) on the **full** table **[normalized Inf.Pos.* | log1p(Inf.Hyp.Rad) | log1p(Inf.Kappa)]**. PCA initialization avoids the default **spectral** embedding, which often warns or falls back to random init on large / ill-conditioned matrices. **`n_jobs=1`** matches `random_state` (umap-learn pins parallelism for reproducibility). The following cells **reuse the same** 2D embedding and only change the visualization (scalar color, hexbin density, degree).

**How to read the output:** Nearby points share similar **combined** features in UMAP space. Comparing **four** views isolates whether radius, κ, degree, or raw density explains what you see. Runtime is dominated by the **single** `fit_transform` above.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import umap

import dmercator3d_io as dm

merged = dm.load_merged_parquet(Path("cache/merged.parquet"))
if "degree" not in merged.columns:
    paths = dm.paths_for_run("d3")
    G = dm.load_edges_graph(paths["edge"])
    deg_map = {str(a): int(b) for a, b in G.degree()}
    merged = merged.assign(degree=merged["Vertex"].astype(str).map(deg_map))
    merged["degree"] = merged["degree"].fillna(0).astype(int)

U = dm.normalize_direction_nd(merged)
X = np.hstack(
    [
        U,
        np.log1p(merged["Inf.Hyp.Rad"]).to_numpy()[:, None],
        np.log1p(merged["Inf.Kappa"]).to_numpy()[:, None],
    ]
)
print("UMAP input:", X.shape)
red = umap.UMAP(
    n_neighbors=30,
    min_dist=0.1,
    random_state=42,
    init="pca",
    n_jobs=1,
).fit_transform(X)
print("embedding:", red.shape)


UMAP input: (17090, 6)


### Scatter — colored by log1p hyperbolic radius

**Method:** Same 2D embedding `red` for everyone. Points colored by `log1p(Inf.Hyp.Rad)`.

**How to read it:** Smooth color gradients suggest radius varies continuously across UMAP neighborhoods; scattered color noise means angular/log-κ dimensions dominate local 2D structure.


In [ ]:
plt.figure(figsize=(6, 5), dpi=200)
plt.scatter(red[:, 0], red[:, 1], s=2, alpha=0.35, c=np.log1p(merged["Inf.Hyp.Rad"]), cmap="viridis")
plt.colorbar(label="log1p Inf.Hyp.Rad")
plt.title("UMAP (dir + log r_H + log κ) — color: log1p hyperbolic radius")
plt.tight_layout()
plt.show()


### Hexbin — density in the UMAP plane

**Method:** Counts how many embedded points fall in each hexagonal bin (no per-point color).

**How to read it:** Hot spots are **overcrowded** regions of the 2D layout; cool gaps are **sparse** areas. Useful when scatter plots saturate (many overlapping points at full *N*).


In [ ]:
plt.figure(figsize=(6, 5), dpi=200)
hb = plt.hexbin(red[:, 0], red[:, 1], gridsize=80, cmap="magma", mincnt=1, linewidths=0)
plt.colorbar(hb, label="points per hex bin")
plt.title("UMAP plane — sampling density (hexbin)")
plt.tight_layout()
plt.show()


### Scatter — colored by log1p κ

**Method:** Same `red`; color is `log1p(Inf.Kappa)` instead of radius.

**How to read it:** Compare to the first figure: if patterns **differ**, κ and hyperbolic radius carry **different** 2D structure relative to direction; if they look **similar**, they may be partially redundant in this UMAP view.


In [ ]:
plt.figure(figsize=(6, 5), dpi=200)
plt.scatter(red[:, 0], red[:, 1], s=2, alpha=0.35, c=np.log1p(merged["Inf.Kappa"]), cmap="plasma")
plt.colorbar(label="log1p Inf.Kappa")
plt.title("Same UMAP — color: log1p κ (popularity-like)")
plt.tight_layout()
plt.show()


### Scatter — colored by PPI degree

**Method:** Same `red`; color is **graph degree** from `merged` (filled from `cache/merged.parquet`, or from `edges_GC.edge` if the column was missing). Display capped at the **99th percentile** so a few megahubs do not wash out the colormap.

**How to read it:** Hubs clustering in pockets suggests degree covaries with this feature-space geometry; a uniform mix suggests degree is not the main axis of this particular 2D projection.


In [ ]:
deg = merged["degree"].to_numpy()
vmax = float(max(np.percentile(deg, 99), 1))
plt.figure(figsize=(6, 5), dpi=200)
sc = plt.scatter(red[:, 0], red[:, 1], s=2, alpha=0.35, c=deg, cmap="inferno", vmin=0, vmax=vmax)
plt.colorbar(sc, label="graph degree")
plt.title(f"Same UMAP — color: degree (display cap p99 ≈ {vmax:.0f})")
plt.tight_layout()
plt.show()
